In [1]:
import numpy as np

# load EMG samples
data = np.load("data/myo_ds_30l_10ol.npz")
X = data['X']

# load labels
labels = np.load("data/myo_ds_30l_10ol_kmeans_labels.npz")
y = labels['y']

print("Data shape:", X.shape)
print("Labels shape:", y.shape)


Data shape: (5595, 30, 8)
Labels shape: (5595,)


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (4476, 30, 8)
Test shape: (1119, 30, 8)


In [3]:
from tensorflow.keras import models, layers
import numpy as np

timesteps = X.shape[1]
channels = X.shape[2]
n_classes = len(np.unique(y))

model = models.Sequential([
    layers.Conv1D(32, 3, activation='relu', input_shape=(timesteps, channels)),
    layers.MaxPooling1D(2),
    layers.Conv1D(64, 3, activation='relu'),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(n_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\Jatinp\imu_project\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)               │ (None, 28, 32)        │          800 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ max_pooling1d (MaxPooling1D)  │ (None, 14, 32)        │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ conv1d_1 (Conv1D)             │ (None, 12, 64)        │        6,208 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ global_average_pooling1d      │ (None, 64)            │            0 │
│ (GlobalAveragePooling1D)      │                       │              │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense (Dense)                 │ (None, 64)            │        4,160 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_1 (Dense)               │ (None, 4)             │          260 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 11,428 (44.64 KB)

 Trainable params: 11,428 (44.64 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.7704 - loss: 6.0859 - val_accuracy: 0.8605 - val_loss: 0.6598
Epoch 2/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.8665 - loss: 0.7041 - val_accuracy: 0.8438 - val_loss: 1.0040
Epoch 3/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8807 - loss: 0.5236 - val_accuracy: 0.8426 - val_loss: 0.6966
Epoch 4/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9017 - loss: 0.3994 - val_accuracy: 0.8772 - val_loss: 0.5434
Epoch 5/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9028 - loss: 0.3395 - val_accuracy: 0.8694 - val_loss: 0.6677
Epoch 6/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9084 - loss: 0.3613 - val_accuracy: 0.9018 - val_loss: 0.3825
Epoch 7/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9070 - loss: 0.3555 - val_accuracy: 0.8828 - val_loss: 0.4340
Epoch 8/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9277 - loss: 0.2620 - val_accuracy:

In [5]:
loss, acc = model.evaluate(X_test, y_test)
print("Test accuracy:", acc * 100, "%")

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9160 - loss: 0.2305
Test accuracy: 91.59964323043823 %
